# Drone Swarm Robotics Simulator
Boids flocking, obstacle avoidance, formation control (Hungarian assignment + fault tolerance) and PSO-based search & rescue.

Run all cells top to bottom (Colab: *Runtime > Run all*).

In [ ]:
# ==============================================================
#  DRONE SWARM ROBOTICS SIMULATOR
#  1) Boids flocking   2) Obstacle avoidance + goal seeking
#  3) Formation control (Hungarian assignment, fault tolerance)
#  4) Search & Rescue with Particle Swarm Optimization (PSO)
# ==============================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import animation, rc
from dataclasses import dataclass
from scipy.optimize import linear_sum_assignment
from scipy.spatial.distance import cdist
from IPython.display import HTML, display

rc('animation', html='jshtml')
plt.rcParams['animation.embed_limit'] = 80   # MB, for long animations
plt.rcParams['figure.dpi'] = 85
np.random.seed(42)
print("Setup done")

## 1. Core: configuration, helpers and building obstacles

In [ ]:
# ==============================================================
#  CORE: config, helpers, buildings (obstacles)
# ==============================================================
@dataclass
class SwarmConfig:
    n_drones: int = 25
    world: tuple = (100.0, 100.0, 50.0)   # x, y, z limits (metres)
    dt: float = 0.1                       # time step (s)
    max_speed: float = 8.0                # m/s
    min_speed: float = 1.5                # flocking drones keep moving
    max_accel: float = 7.0                # m/s^2 (motor limit)
    # Boids (Reynolds 1987)
    neighbor_radius: float = 15.0
    separation_radius: float = 4.5
    w_sep: float = 2.0
    w_ali: float = 1.0
    w_coh: float = 0.7
    # Goal / obstacles / boundaries
    w_goal: float = 1.3
    w_obs: float = 5.0
    obs_margin: float = 7.0
    w_bound: float = 3.0
    bound_margin: float = 6.0
    min_altitude: float = 3.0
    # Formation PD gains
    kp: float = 1.0
    kd: float = 2.0
    formation_speed: float = 4.0          # speed of the formation centre (m/s)


@dataclass
class Building:
    """Vertical cylinder obstacle (tower / tree / pole)."""
    x: float
    y: float
    r: float
    h: float


def clip_norm(v, max_norm):
    n = np.linalg.norm(v, axis=-1, keepdims=True)
    return v * np.minimum(1.0, max_norm / (n + 1e-9))


def unit(v):
    return v / (np.linalg.norm(v, axis=-1, keepdims=True) + 1e-9)


def make_city(seed=3, n=7, world=(100, 100, 50), keep_clear=((12, 12), (80, 75), (20, 85))):
    """Random 'city' of buildings, keeping start / goal zones free."""
    rng = np.random.default_rng(seed)
    blds = []
    while len(blds) < n:
        x, y = rng.uniform(20, world[0] - 15, 2)
        r, h = rng.uniform(4, 7), rng.uniform(20, 42)
        if any(np.hypot(x - cx, y - cy) < r + 14 for cx, cy in keep_clear):
            continue
        if any(np.hypot(x - b.x, y - b.y) < r + b.r + 8 for b in blds):
            continue
        blds.append(Building(x, y, r, h))
    return blds

print("Core ready")

## 2. Formation shapes

In [ ]:
# ==============================================================
#  FORMATION SHAPES  (each returns n x 3 offsets around a centre)
# ==============================================================
def formation_circle(n, radius=14.0):
    a = np.linspace(0, 2 * np.pi, n, endpoint=False)
    return np.c_[radius * np.cos(a), radius * np.sin(a), np.zeros(n)]

def formation_v(n, spacing=4.0):
    off = np.zeros((n, 3))
    for i in range(1, n):
        k, side = (i + 1) // 2, (1 if i % 2 else -1)
        off[i] = [-k * spacing, side * k * spacing, 0]
    return off - off.mean(0)

def formation_grid(n, spacing=5.0):
    c = int(np.ceil(np.sqrt(n)))
    pts = np.array([[i % c, i // c, 0] for i in range(n)], float) * spacing
    return pts - pts.mean(0)

def formation_helix(n, radius=10.0, turns=2.0, height=18.0):
    t = np.linspace(0, 2 * np.pi * turns, n)
    off = np.c_[radius * np.cos(t), radius * np.sin(t), np.linspace(-height / 2, height / 2, n)]
    return off

def formation_sphere(n, radius=12.0):
    i = np.arange(n) + 0.5                      # Fibonacci sphere
    phi = np.arccos(1 - 2 * i / n)
    th = np.pi * (1 + 5 ** 0.5) * i
    return radius * np.c_[np.cos(th) * np.sin(phi), np.sin(th) * np.sin(phi), np.cos(phi)]

FORMATIONS = {'circle': formation_circle, 'v': formation_v, 'grid': formation_grid,
              'helix': formation_helix, 'sphere': formation_sphere}

# quick preview of all shapes
fig = plt.figure(figsize=(14, 3))
for k, (name, fn) in enumerate(FORMATIONS.items()):
    ax = fig.add_subplot(1, 5, k + 1, projection='3d')
    P = fn(25)
    ax.scatter(*P.T, s=18, c='tab:blue')
    ax.set_title(name); ax.set_xticks([]); ax.set_yticks([]); ax.set_zticks([])
plt.tight_layout(); plt.show()

## 3. DroneSwarm class
Point-mass quadrotor model with two modes: **flock** (Boids + goal seeking) and **formation** (PD control to Hungarian-assigned slots). Building avoidance, boundary walls and altitude limits are always active.

In [ ]:
# ==============================================================
#  DRONE SWARM  (vectorised point-mass quadrotor model)
#  modes: 'flock'  -> Boids + goal seeking
#         'formation' -> PD control to Hungarian-assigned slots
#  always on: building avoidance, boundary walls, min altitude
# ==============================================================
class DroneSwarm:
    def __init__(self, cfg: SwarmConfig, start_center=(12, 12, 8), spread=6.0, seed=0):
        self.cfg, self.rng = cfg, np.random.default_rng(seed)
        n = cfg.n_drones
        self.W = np.array(cfg.world, float)
        self.pos = np.array(start_center, float) + self.rng.normal(0, spread, (n, 3))
        self.pos[:, 2] = np.clip(self.pos[:, 2], cfg.min_altitude, None)
        self.vel = self.rng.normal(0, 1.0, (n, 3))
        self.battery = np.full(n, 100.0)
        self.alive = np.ones(n, bool)
        self.buildings, self.goal = [], None
        self.mode, self.phase = 'flock', 'start'
        self.form_fn = self.form_off = self.form_center = self.form_goal = None
        self.t = 0.0
        self.traj, self.alive_traj, self.target_traj, self.log = [], [], [], []
        self.building_hits = 0
        self._record(self._pairwise()[1])

    # ---------- neighbourhood ----------
    def _pairwise(self):
        diff = self.pos[:, None, :] - self.pos[None, :, :]          # i - j
        dist = np.linalg.norm(diff, axis=-1)
        np.fill_diagonal(dist, np.inf)
        dist[~self.alive, :] = np.inf
        dist[:, ~self.alive] = np.inf
        return diff, dist

    # ---------- Boids rules ----------
    def separation(self, diff, dist, radius=None):
        r = radius or self.cfg.separation_radius
        w = np.where(dist < r, 1.0 / (dist ** 2 + 1e-6), 0.0)
        push = (diff * w[..., None]).sum(1)
        has = (dist < r).any(1)[:, None]
        steer = unit(push) * self.cfg.max_speed - self.vel
        return np.where(has, clip_norm(steer, self.cfg.max_accel), 0.0)

    def alignment(self, dist):
        nb = dist < self.cfg.neighbor_radius
        cnt = nb.sum(1, keepdims=True)
        avg = (nb[..., None] * self.vel[None]).sum(1) / np.maximum(cnt, 1)
        steer = unit(avg) * self.cfg.max_speed - self.vel
        return np.where(cnt > 0, clip_norm(steer, self.cfg.max_accel), 0.0)

    def cohesion(self, dist):
        nb = dist < self.cfg.neighbor_radius
        cnt = nb.sum(1, keepdims=True)
        center = (nb[..., None] * self.pos[None]).sum(1) / np.maximum(cnt, 1)
        steer = unit(center - self.pos) * self.cfg.max_speed - self.vel
        return np.where(cnt > 0, clip_norm(steer, self.cfg.max_accel), 0.0)

    def seek(self, target, slow_radius=20.0):
        d = target - self.pos
        dn = np.linalg.norm(d, axis=1, keepdims=True)
        desired = unit(d) * self.cfg.max_speed * np.minimum(1.0, dn / slow_radius)
        return clip_norm(desired - self.vel, self.cfg.max_accel)

    # ---------- obstacle avoidance (potential field + tangential slide) ----------
    def avoid_buildings(self):
        acc = np.zeros_like(self.pos)
        m = self.cfg.obs_margin
        for b in self.buildings:
            h = self.pos[:, :2] - [b.x, b.y]
            d = np.linalg.norm(h, axis=1)
            surf = d - b.r
            below = self.pos[:, 2] < b.h + 3
            near = (surf < m) & below
            if not near.any():
                continue
            strength = np.where(near, ((m - np.maximum(surf, 0.2)) / m) ** 2, 0.0)
            radial = unit(h)
            tang = np.c_[-radial[:, 1], radial[:, 0]]
            sgn = np.sign((tang * self.vel[:, :2]).sum(1, keepdims=True) + 1e-6)
            acc[:, :2] += (radial * 3.0 + tang * sgn * 1.5) * strength[:, None] * self.cfg.max_accel
            self.building_hits += int(((surf < 0) & below & self.alive).sum())
        return acc

    def boundaries(self):
        m, acc = self.cfg.bound_margin, np.zeros_like(self.pos)
        lo = np.array([0, 0, self.cfg.min_altitude])
        acc += np.clip((lo + m - self.pos) / m, 0, None)
        acc -= np.clip((self.pos - (self.W - m)) / m, 0, None)
        return acc * self.cfg.max_accel

    # ---------- formation control ----------
    def set_formation(self, shape, center=None, goal=None, **kw):
        """shape: name in FORMATIONS or callable(n)->offsets. Uses Hungarian assignment."""
        self.form_fn = (lambda n: FORMATIONS[shape](n, **kw)) if isinstance(shape, str) else shape
        self.form_center = np.array(center if center is not None else self.pos[self.alive].mean(0), float)
        self.form_goal = None if goal is None else np.array(goal, float)
        self._assign()
        self.mode = 'formation'

    def _assign(self):
        idx = np.where(self.alive)[0]
        offs = self.form_fn(len(idx))
        R = self._heading_rot()                     # assign slots in the rotated frame
        cost = cdist(self.pos[idx], self.form_center + offs @ R.T)
        r, c = linear_sum_assignment(cost)          # minimise total travel distance
        self.form_off = np.zeros_like(self.pos)
        self.form_off[idx[r]] = offs[c]

    def _heading_rot(self):
        if self.form_goal is None:
            return np.eye(3)
        d = self.form_goal[:2] - self.form_center[:2]
        if np.linalg.norm(d) < 1e-3:
            return getattr(self, '_last_R', np.eye(3))
        a = np.arctan2(d[1], d[0])
        self._last_R = np.array([[np.cos(a), -np.sin(a), 0], [np.sin(a), np.cos(a), 0], [0, 0, 1]])
        return self._last_R

    def targets(self):
        return self.form_center + self.form_off @ self._heading_rot().T

    def formation_control(self):
        c = self.cfg
        if self.form_goal is not None:                       # move the whole formation
            d = self.form_goal - self.form_center
            dn = np.linalg.norm(d)
            if dn > 1e-3:
                self.form_center = self.form_center + d / dn * min(c.formation_speed * c.dt, dn)
        e = self.targets() - self.pos
        desired = clip_norm(c.kp * e, c.max_speed)
        return clip_norm(c.kd * (desired - self.vel), c.max_accel)

    # ---------- fault tolerance ----------
    def fail_drones(self, k):
        alive_idx = np.where(self.alive)[0]
        dead = self.rng.choice(alive_idx, size=min(k, len(alive_idx) - 1), replace=False)
        self.alive[dead] = False
        if self.mode == 'formation':
            self._assign()                                  # re-plan with the survivors
        return dead

    # ---------- simulation step ----------
    def step(self):
        c = self.cfg
        diff, dist = self._pairwise()
        acc = np.zeros_like(self.pos)
        if self.mode == 'flock':
            acc += c.w_sep * self.separation(diff, dist) + c.w_ali * self.alignment(dist) \
                 + c.w_coh * self.cohesion(dist)
            if self.goal is not None:
                acc += c.w_goal * self.seek(self.goal)
        else:
            acc += self.formation_control() + 1.5 * self.separation(diff, dist, radius=2.5)
        acc += c.w_obs * self.avoid_buildings() + c.w_bound * self.boundaries()
        acc = clip_norm(acc, c.max_accel * 1.5)

        self.vel = clip_norm(self.vel + acc * c.dt, c.max_speed)
        if self.mode == 'flock' and self.goal is None:      # birds never hover
            sp = np.linalg.norm(self.vel, axis=1, keepdims=True)
            self.vel = np.where(sp < c.min_speed, unit(self.vel) * c.min_speed, self.vel)
        # failed drones: controlled emergency landing
        self.vel[~self.alive] = [0, 0, -3.0]
        self.pos += self.vel * c.dt
        self.pos[:, 2] = np.maximum(self.pos[:, 2], 0)
        self.vel[~self.alive & (self.pos[:, 2] <= 0)] = 0

        speed = np.linalg.norm(self.vel, axis=1)
        self.battery -= np.where(self.alive, (0.08 + 0.002 * speed ** 2) * c.dt, 0)  # % per step
        self.t += c.dt
        self._record(dist)

    def run(self, steps, phase=None, until=None):
        if phase: self.phase = phase
        for _ in range(steps):
            self.step()
            if until is not None and until(self):
                break
        return self

    # ---------- telemetry ----------
    def _record(self, dist):
        a = self.alive
        P, V = self.pos[a], self.vel[a]
        sp = np.linalg.norm(V, axis=1)
        cen = P.mean(0)
        d = dist[np.ix_(a, a)]
        row = dict(t=round(self.t, 2), phase=self.phase, alive=int(a.sum()),
                   polarization=float(np.linalg.norm(unit(V).mean(0))),
                   cohesion=float(np.linalg.norm(P - cen, axis=1).mean()),
                   min_separation=float(d.min()) if d.size > 1 else np.nan,
                   near_misses=int((d[np.triu_indices(len(d), 1)] < 1.0).sum()),
                   mean_speed=float(sp.mean()), battery=float(self.battery[a].mean()),
                   goal_dist=float(np.linalg.norm(cen - self.goal)) if (self.goal is not None and self.mode == 'flock') else np.nan,
                   formation_error=float(np.linalg.norm(self.targets()[a] - P, axis=1).mean()) if self.mode == 'formation' else np.nan)
        self.log.append(row)
        self.traj.append(self.pos.copy())
        self.alive_traj.append(self.alive.copy())
        self.target_traj.append(self.targets().copy() if self.mode == 'formation' else None)

    def df(self):
        return pd.DataFrame(self.log)

print("DroneSwarm class ready")

## 4. Visualisation helpers

In [ ]:
# ==============================================================
#  VISUALISATION: 3D animation + telemetry dashboard
# ==============================================================
def draw_building(ax, b, alpha=0.30):
    th, z = np.linspace(0, 2 * np.pi, 20), np.array([0, b.h])
    TH, Z = np.meshgrid(th, z)
    ax.plot_surface(b.x + b.r * np.cos(TH), b.y + b.r * np.sin(TH), Z,
                    color='dimgray', alpha=alpha, linewidth=0)

def animate_swarm(sw, every=4, trail=10, title='Drone Swarm', rotate=True, goals=()):
    traj = np.array(sw.traj)[::every]
    alive = np.array(sw.alive_traj)[::every]
    tgts = sw.target_traj[::every]
    phases = [r['phase'] for r in sw.log][::every]
    n = traj.shape[1]
    fig = plt.figure(figsize=(8, 6.5))
    ax = fig.add_subplot(111, projection='3d')
    ax.set_xlim(0, sw.W[0]); ax.set_ylim(0, sw.W[1]); ax.set_zlim(0, sw.W[2])
    ax.set_xlabel('x (m)'); ax.set_ylabel('y (m)'); ax.set_zlabel('z (m)')
    for b in sw.buildings:
        draw_building(ax, b)
    for g in goals:
        ax.scatter(*g, marker='*', s=250, c='gold', edgecolors='k', zorder=5)
    sc = ax.scatter(*traj[0].T, s=22, c='tab:blue', depthshade=False)
    tg = ax.scatter([], [], [], s=12, marker='x', c='tab:orange', alpha=0.6)
    lines = [ax.plot([], [], [], lw=0.8, alpha=0.5, c='tab:blue')[0] for _ in range(n)]
    txt = ax.text2D(0.02, 0.95, '', transform=ax.transAxes, fontsize=10,
                    bbox=dict(fc='white', alpha=0.8))
    ax.set_title(title)

    def update(f):
        P = traj[f]
        sc._offsets3d = (P[:, 0], P[:, 1], P[:, 2])
        sc.set_color(np.where(alive[f], 'tab:blue', 'red'))
        T = tgts[f]
        tg._offsets3d = (T[:, 0], T[:, 1], T[:, 2]) if T is not None else ([], [], [])
        for i, l in enumerate(lines):
            seg = traj[max(0, f - trail):f + 1, i]
            l.set_data(seg[:, 0], seg[:, 1]); l.set_3d_properties(seg[:, 2])
        txt.set_text(f"t = {f * every * sw.cfg.dt:5.1f}s | {phases[f]} | alive {alive[f].sum()}/{n}")
        if rotate:
            ax.view_init(elev=28, azim=-60 + 0.4 * f)
        return [sc]

    anim = animation.FuncAnimation(fig, update, frames=len(traj), interval=60)
    plt.close(fig)
    return HTML(anim.to_jshtml())

def plot_metrics(sw, title='Swarm telemetry'):
    df = sw.df().iloc[1:]          # skip t=0 snapshot
    cols = [('polarization', 'Polarization (1 = all aligned)'), ('cohesion', 'Mean dist. to centroid (m)'),
            ('min_separation', 'Min drone-drone distance (m)'), ('mean_speed', 'Mean speed (m/s)'),
            ('formation_error', 'Formation error (m)'), ('battery', 'Mean battery (%)')]
    fig, axs = plt.subplots(2, 3, figsize=(15, 6.5))
    for ax, (c, lab) in zip(axs.flat, cols):
        for ph, g in df.groupby('phase', sort=False):
            ax.plot(g.t, g[c], lw=1.5, label=ph)
        ax.set_title(lab, fontsize=10); ax.set_xlabel('t (s)'); ax.grid(alpha=0.3)
    axs[0, 2].axhline(1.0, c='r', ls='--', lw=1, label='collision limit')
    axs[0, 0].legend(fontsize=8)
    fig.suptitle(title, fontsize=13); plt.tight_layout(); plt.show()
    summary = df.groupby('phase', sort=False).agg(
        duration_s=('t', lambda s: s.max() - s.min()), min_sep_m=('min_separation', 'min'),
        near_misses=('near_misses', 'sum'), avg_polarization=('polarization', 'mean'),
        final_form_err_m=('formation_error', 'last'), battery_end=('battery', 'last')).round(2)
    display(summary)
    print(f"Building collisions: {sw.building_hits}")

print("Visualisation helpers ready")

## Demo 1 - Boids flocking

In [ ]:
# ==============================================================
#  DEMO 1: Pure Boids flocking (separation + alignment + cohesion)
# ==============================================================
cfg = SwarmConfig(n_drones=30)
flock = DroneSwarm(cfg, start_center=(50, 50, 25), spread=15, seed=1)
flock.vel = flock.rng.normal(0, 4, (cfg.n_drones, 3))       # start with random headings
flock.run(350, phase='boids flocking')

plot_metrics(flock, 'Demo 1 - Boids flocking')
animate_swarm(flock, every=5, title='Demo 1: Boids flocking')

## Demo 2 - Obstacle avoidance + goal seeking

In [ ]:
# ==============================================================
#  DEMO 2: Flock through a city to a goal, avoiding buildings
# ==============================================================
city = make_city(seed=3, n=7)
GOAL = np.array([80.0, 75.0, 20.0])

nav = DroneSwarm(SwarmConfig(n_drones=25), start_center=(12, 12, 10), seed=2)
nav.buildings, nav.goal = city, GOAL
reached = lambda s: np.linalg.norm(s.pos[s.alive].mean(0) - GOAL) < 6
nav.run(700, phase='navigate + avoid', until=reached)
print(f"Swarm reached goal in {nav.t:.1f}s")

plot_metrics(nav, 'Demo 2 - Obstacle avoidance')
animate_swarm(nav, every=5, title='Demo 2: Obstacle avoidance + goal seeking', goals=[GOAL])

## Demo 3 - Formation control and fault tolerance

In [ ]:
# ==============================================================
#  DEMO 3: Formation control + fault tolerance
#  grid -> circle -> sphere -> (5 drones fail, auto re-assign) -> V moving
# ==============================================================
fm = DroneSwarm(SwarmConfig(n_drones=25), start_center=(20, 20, 5), spread=5, seed=4)
C = (45, 50, 25)
fm.set_formation('grid', center=C);   fm.run(150, phase='grid')
fm.set_formation('circle', center=C); fm.run(150, phase='circle')
fm.set_formation('sphere', center=C); fm.run(150, phase='sphere')
dead = fm.fail_drones(5)
print("Failed drones:", sorted(dead.tolist()), "-> formation re-planned with", fm.alive.sum(), "drones")
fm.run(120, phase='sphere (5 failed)')
fm.set_formation('v', center=fm.pos[fm.alive].mean(0), goal=(85, 85, 30)); fm.run(250, phase='V moving')

plot_metrics(fm, 'Demo 3 - Formation control')
animate_swarm(fm, every=5, title='Demo 3: Formation control (x = target slots)')

## Experiment - separation weight vs safety

In [ ]:
# ==============================================================
#  EXPERIMENT: effect of separation weight & swarm size on safety
# ==============================================================
rows = []
for n in [15, 30, 50]:
    for w_sep in [0.0, 0.5, 1.0, 2.0, 4.0]:
        for seed in range(3):
            s = DroneSwarm(SwarmConfig(n_drones=n, w_sep=w_sep), start_center=(50, 50, 25), spread=12, seed=seed)
            s.run(250)
            d = s.df().iloc[50:]                          # ignore transient
            rows.append(dict(n=n, w_sep=w_sep, min_sep=d.min_separation.min(),
                             near_misses=d.near_misses.sum(), polarization=d.polarization.mean(),
                             cohesion=d.cohesion.mean()))
exp = pd.DataFrame(rows).groupby(['n', 'w_sep']).mean().round(2)
display(exp)

fig, axs = plt.subplots(1, 3, figsize=(15, 4))
for n, g in exp.reset_index().groupby('n'):
    axs[0].plot(g.w_sep, g.min_sep, 'o-', label=f'N={n}')
    axs[1].plot(g.w_sep, g.near_misses, 'o-', label=f'N={n}')
    axs[2].plot(g.w_sep, g.polarization, 'o-', label=f'N={n}')
for ax, t in zip(axs, ['Min separation (m)', 'Near-misses (<1 m)', 'Polarization']):
    ax.set_title(t); ax.set_xlabel('separation weight'); ax.grid(alpha=.3); ax.legend()
plt.tight_layout(); plt.show()

## 5. Search & Rescue (PSO vs lawnmower vs random)

In [ ]:
# ==============================================================
#  SEARCH & RESCUE  (2D, drones at fixed altitude)
#  Survivors emit a weak noisy signal (e.g. phone RF).  Strategies:
#   'pso'       : PSO on signal + frontier exploration when signal is weak
#   'lawnmower' : classic boustrophedon sweep, strips split between drones
#   'random'    : random waypoint
# ==============================================================
class SearchAndRescue:
    def __init__(self, n_drones=12, area=100, n_survivors=5, strategy='pso', seed=0,
                 sensor_r=5.0, sigma=14.0, noise=0.03, max_speed=5.0, dt=0.5, comm_r=25.0):
        self.rng = np.random.default_rng(seed)
        self.n, self.A, self.strategy = n_drones, area, strategy
        self.sensor_r, self.detect_r, self.sigma, self.noise = sensor_r, sensor_r, sigma, noise
        self.comm_r = comm_r
        self.vmax, self.dt, self.t = max_speed, dt, 0.0
        self.pos = np.array([5.0, 5.0]) + self.rng.uniform(0, 4, (n_drones, 2))
        self.vel = np.zeros((n_drones, 2))
        self.surv = self.rng.uniform(10, area - 10, (n_survivors, 2))
        self.found = np.zeros(n_survivors, bool)
        self.found_t = np.full(n_survivors, np.nan)
        self.cov = np.zeros((area, area), bool)       # seen by camera (sensor_r)
        self.heard = np.zeros((area, area), bool)     # listened by RF receiver (hear_r) -> PSO frontier map
        self.hear_r = 0.7 * sigma * np.sqrt(2 * np.log(1 / 0.15))   # radius where signal > threshold
        gx, gy = np.meshgrid(np.arange(area) + .5, np.arange(area) + .5)
        self.cells = np.c_[gx.ravel(), gy.ravel()]
        # PSO memory
        self.pb_pos, self.pb_val = self.pos.copy(), self.signal(self.pos)
        self.explore_tgt = np.full((n_drones, 2), np.nan)
        if strategy == 'lawnmower':
            self.paths = self._lawnmower_paths(); self.wp = np.zeros(n_drones, int)
        self.traj, self.cov_hist, self.found_hist = [], [], []
        self._update_coverage(); self._save()

    # ---- sensing ----
    def true_signal(self, P):
        act = self.surv[~self.found]
        if len(act) == 0:
            return np.zeros(len(P))
        return np.exp(-cdist(P, act) ** 2 / (2 * self.sigma ** 2)).max(1)

    def signal(self, P):
        return self.true_signal(P) + self.rng.normal(0, self.noise, len(P))

    def _update_coverage(self):
        d = cdist(self.cells, self.pos).min(1)
        self.cov.ravel()[d < self.sensor_r] = True
        self.heard.ravel()[d < self.hear_r] = True

    # ---- strategies ----
    def _lawnmower_paths(self):
        w = 2 * self.sensor_r * 0.95
        xs = np.arange(w / 2, self.A, w)
        strips = [np.array([[x, 3], [x, self.A - 3]] if k % 2 == 0 else [[x, self.A - 3], [x, 3]]) for k, x in enumerate(xs)]
        paths = [[] for _ in range(self.n)]
        for k, s in enumerate(strips):
            paths[k % self.n].extend(s)
        return [np.array(p) for p in paths]

    def _pick_frontier(self, i):
        free = self.cells[~self.heard.ravel()]
        if len(free) == 0:
            return self.rng.uniform(0, self.A, 2)
        cand = free[self.rng.choice(len(free), min(150, len(free)), replace=False)]
        others = np.delete(np.r_[self.pos, np.nan_to_num(self.explore_tgt, nan=-1e3)], [i, self.n + i], 0)
        score = -np.linalg.norm(cand - self.pos[i], axis=1) + 0.6 * cdist(cand, others).min(1)
        return cand[score.argmax()]

    def _desired_velocity(self):
        v = np.zeros_like(self.pos)
        if self.strategy == 'pso':
            s = self.signal(self.pos)
            better = s > self.pb_val
            self.pb_pos[better], self.pb_val[better] = self.pos[better], s[better]
            D = cdist(self.pos, self.pos)
            for i in range(self.n):
                nb = np.where(D[i] < self.comm_r)[0]            # only drones in radio range share info
                j = nb[self.pb_val[nb].argmax()]
                if self.pb_val[j] > 0.15:                       # signal heard -> PSO converge on source
                    r1, r2 = self.rng.random(2)
                    v[i] = 0.5 * self.vel[i] + 1.2 * r1 * (self.pb_pos[i] - self.pos[i]) \
                           + 1.5 * r2 * (self.pb_pos[j] - self.pos[i])
                    # turbulence: keep probing around the best point (bigger when signal is weak)
                    v[i] += self.rng.normal(0, 1, 2) * self.vmax * (1.0 - min(self.pb_val[j], 1.0)) * 1.5
                else:                                           # silence -> explore unvisited frontier
                    t = self.explore_tgt[i]
                    if np.isnan(t).any() or np.linalg.norm(t - self.pos[i]) < 2 or self.heard[int(t[1]), int(t[0])]:
                        self.explore_tgt[i] = t = self._pick_frontier(i)
                    v[i] = unit(t - self.pos[i]) * self.vmax
        elif self.strategy == 'lawnmower':
            for i in range(self.n):
                p = self.paths[i]
                if len(p) == 0 or self.wp[i] >= len(p):        # sweep done -> random patrol
                    t = self.explore_tgt[i]
                    if np.isnan(t).any() or np.linalg.norm(t - self.pos[i]) < 2:
                        self.explore_tgt[i] = t = self.rng.uniform(0, self.A, 2)
                    v[i] = unit(t - self.pos[i]) * self.vmax
                    continue
                d = p[self.wp[i]] - self.pos[i]
                if np.linalg.norm(d) < 1.5: self.wp[i] += 1
                v[i] = unit(d) * self.vmax
        else:  # random waypoint
            for i in range(self.n):
                t = self.explore_tgt[i]
                if np.isnan(t).any() or np.linalg.norm(t - self.pos[i]) < 2:
                    self.explore_tgt[i] = t = self.rng.uniform(0, self.A, 2)
                v[i] = unit(t - self.pos[i]) * self.vmax
        # inter-drone repulsion (collision avoidance)
        diff = self.pos[:, None] - self.pos[None]
        d = np.linalg.norm(diff, axis=-1); np.fill_diagonal(d, np.inf)
        v += (diff * np.where(d < 3, 4 / (d ** 2 + .1), 0)[..., None]).sum(1)
        return clip_norm(v, self.vmax)

    # ---- simulation ----
    def step(self):
        self.vel = self._desired_velocity()
        self.pos = np.clip(self.pos + self.vel * self.dt, 0, self.A - 1e-3)
        self.t += self.dt
        self._update_coverage()
        hit = (cdist(self.surv, self.pos).min(1) < self.detect_r) & ~self.found
        if hit.any():
            self.found[hit], self.found_t[hit] = True, self.t
            self.pb_pos, self.pb_val = self.pos.copy(), self.signal(self.pos)   # field changed -> reset memory
        self._save()

    def _save(self):
        self.traj.append(self.pos.copy()); self.cov_hist.append(self.cov.mean() * 100)
        self.found_hist.append(self.found.copy())

    def run(self, max_steps=1200):
        while not self.found.all() and len(self.traj) <= max_steps:
            self.step()
        return dict(strategy=self.strategy, all_found=bool(self.found.all()),
                    time_all_s=self.t if self.found.all() else np.nan,
                    mean_find_time_s=np.nanmean(self.found_t), coverage_pct=self.cov.mean() * 100)

print("SearchAndRescue ready")

## Demo 4 - Search & Rescue animation

In [ ]:
# ==============================================================
#  DEMO 4: Search & Rescue animation (PSO swarm)
# ==============================================================
def animate_sar(sar, every=3, trail=12):
    idx = list(range(0, len(sar.traj), every))
    if idx[-1] != len(sar.traj) - 1: idx.append(len(sar.traj) - 1)     # always show final frame
    traj, fh = np.array(sar.traj)[idx], [sar.found_hist[k] for k in idx]
    gx, gy = np.meshgrid(np.linspace(0, sar.A, 80), np.linspace(0, sar.A, 80))
    act = sar.surv
    field = np.exp(-cdist(np.c_[gx.ravel(), gy.ravel()], act) ** 2 / (2 * sar.sigma ** 2)).max(1).reshape(gx.shape)
    # rebuild coverage frame by frame
    covs, cov = [], np.zeros((sar.A, sar.A), bool)
    full = np.array(sar.traj)
    for k in range(len(full)):
        cov.ravel()[cdist(sar.cells, full[k]).min(1) < sar.sensor_r] = True
        if k in idx: covs.append(cov.copy())

    fig, ax = plt.subplots(figsize=(7, 7))
    im = ax.imshow(covs[0], origin='lower', extent=(0, sar.A, 0, sar.A), cmap='Greens', vmin=0, vmax=2.5)
    ax.contour(gx, gy, field, levels=6, cmap='Reds', alpha=0.35)
    sv = ax.scatter(*act.T, marker='X', s=140, c='red', edgecolors='k', zorder=4)
    dr = ax.scatter(*traj[0].T, s=35, c='tab:blue', edgecolors='k', zorder=5)
    lines = [ax.plot([], [], lw=0.8, c='tab:blue', alpha=.5)[0] for _ in range(sar.n)]
    txt = ax.set_title('')
    def update(f):
        dr.set_offsets(traj[f]); im.set_data(covs[f])
        sv.set_color(np.where(fh[f], 'lime', 'red'))
        for i, l in enumerate(lines):
            seg = traj[max(0, f - trail):f + 1, i]; l.set_data(seg[:, 0], seg[:, 1])
        txt.set_text(f"{sar.strategy.upper()} | t={idx[f] * sar.dt:.0f}s | found {fh[f].sum()}/{len(fh[f])} | coverage {sar.cov_hist[idx[f]]:.0f}%")
        return [dr]
    anim = animation.FuncAnimation(fig, update, frames=len(traj), interval=70)
    plt.close(fig)
    return HTML(anim.to_jshtml())

sar = SearchAndRescue(n_drones=10, area=150, n_survivors=5, strategy="pso", seed=7)
res = sar.run()
print(res)
animate_sar(sar)

## Benchmark - search strategies

In [ ]:
# ==============================================================
#  BENCHMARK: PSO vs Lawnmower vs Random  (8 random worlds each)
#  Scenario A: small area, many drones | Scenario B: large area, few drones
# ==============================================================
SCENARIOS = {'A: 100x100 m, 12 drones': dict(area=100, n_drones=12, n_survivors=5),
             'B: 200x200 m, 10 drones': dict(area=200, n_drones=10, n_survivors=6)}
results, curves = [], {}
for sc, kw in SCENARIOS.items():
    for strat in ['pso', 'lawnmower', 'random']:
        curves[(sc, strat)] = []
        for seed in range(8):
            s = SearchAndRescue(strategy=strat, seed=seed, **kw)
            results.append(dict(scenario=sc, **s.run(max_steps=3000)))
            curves[(sc, strat)].append(s.cov_hist)
bench = pd.DataFrame(results)
summary = bench.groupby(['scenario', 'strategy']).agg(
    success_rate=('all_found', 'mean'), time_all_mean_s=('time_all_s', 'mean'),
    time_all_std_s=('time_all_s', 'std'), mean_find_time_s=('mean_find_time_s', 'mean'),
    coverage_pct=('coverage_pct', 'mean')).round(1)
display(summary)

fig, axs = plt.subplots(1, 3, figsize=(17, 4.5))
summary['time_all_mean_s'].unstack().plot.bar(ax=axs[0], rot=0, yerr=summary['time_all_std_s'].unstack(), capsize=4)
axs[0].set_title('Time to find ALL survivors (lower = better)'); axs[0].set_ylabel('s'); axs[0].set_xlabel('')
for ax, sc in zip(axs[1:], SCENARIOS):
    for strat in ['pso', 'lawnmower', 'random']:
        cs = curves[(sc, strat)]; L = max(map(len, cs))
        M = np.array([np.pad(c, (0, L - len(c)), mode='edge') for c in cs])
        ax.plot(np.arange(L) * 0.5, M.mean(0), label=strat)
    ax.set_title(f'Camera coverage - {sc}', fontsize=10); ax.set_xlabel('t (s)'); ax.set_ylabel('% area seen')
    ax.legend(); ax.grid(alpha=.3)
plt.tight_layout(); plt.show()
print("Insight: lawnmower wins when drones are dense; PSO (signal-guided) wins when the area is large and drones are few.")

## Full mission
Takeoff → flock through city → circle survey → 4 drones fail → helix → V formation to landing zone.

In [ ]:
# ==============================================================
#  FULL MISSION: takeoff -> flock through city -> circle survey
#  -> 4 drones fail -> helix -> V formation flies to landing zone
# ==============================================================
city = make_city(seed=3, n=7)
WP1, WP2 = np.array([80.0, 75.0, 22.0]), np.array([30.0, 60.0, 30.0])

m = DroneSwarm(SwarmConfig(n_drones=24), seed=11)
m.pos = np.c_[formation_grid(24, 4.0)[:, :2] + 14, np.zeros(24)]   # drones parked on launch pad
m.vel[:] = 0; m.traj[0] = m.pos.copy()
m.buildings = city
m.set_formation('grid', center=(12, 12, 12), spacing=3.5); m.run(80, phase='1 takeoff (grid)')
m.mode, m.goal = 'flock', WP1
m.run(700, phase='2 flock to WP1', until=lambda s: np.linalg.norm(s.pos[s.alive].mean(0) - WP1) < 6)
m.goal = None
m.set_formation('circle', center=WP1, radius=13); m.run(150, phase='3 circle survey')
dead = m.fail_drones(4);                          m.run(100, phase='4 failure + re-plan')
m.set_formation('helix', center=WP1);             m.run(150, phase='5 helix')
m.set_formation('v', center=m.pos[m.alive].mean(0), goal=WP2, spacing=2.5)
m.run(500, phase='6 V to landing zone', until=lambda s: np.linalg.norm(s.form_center - WP2) < 0.5)
m.run(60, phase='6 V to landing zone')

print(f"Mission time: {m.t:.1f}s | drones lost: {len(dead)} | building collisions: {m.building_hits}")
plot_metrics(m, 'Full mission telemetry')
animate_swarm(m, every=6, trail=8, title='Full drone-swarm mission', goals=[WP1, WP2])